# PE6201 · A2 — 脚手架导览 · **问题 A**

## 这个 notebook 是什么

**只跟一个案例，从头走到尾，每一步都打印真实数值。**

案例是 **`CLM-8842`** —— 一道问题 A 理赔，也是 brief 附录 A 中「部分可赔」的演算例子。下面每一个 cell 都只用这一条理赔。凡是写死的值 —— `'CLM-8842'`、`'M-2214'`、`'POL-3310'`、某个手术编码 —— 都来自这条记录，markdown 会标明出处。

**模型是模拟的，数据是真的。** 后端按固定动作序列回放（`backends.py` 里的 `SCRIPTS['CLM-8842']`），所以这次运行确定、免费。底下的工具是在对随包 JSON 做真实查询 —— 除了模型的决策，没有别的是假的。

> **在做问题 B？** 请改用 `A2_Scaffold_Tour_ProblemB.ipynb`。同样九步、同一套脚手架，只是案例不同。你只需打开自己选题对应的那一份。

## 这个 notebook 不是什么

**它本身不含任何业务逻辑。** 每个 cell 都从旁边的 `.py` 文件 import。

> **Notebook 用来探索。真正交付的是模块。**

六个人可以同时改六个模块。六个人改同一个 notebook 会产生合并冲突和无法阅读的 diff —— 而 brief 第 8 节会用你的 commit 历史来核对谁做了什么。

**要提交的是模块和 `run_eval.py`，不是这个 notebook。** D5(a) 写明：阅卷人会 clone 你的仓库并运行。那一步是 `python3 run_eval.py`；notebook 不算。

---
### 在 Colab 里运行
把 `A2_scaffold/` 和 `A2_reference_data/` 上传到 Drive 里同一个文件夹，然后改 setup cell 里的两条路径。本地运行：按顺序执行各个 cell 即可。


## Cell 1 · 环境设置

把脚手架加入 import 路径，**切换到问题 A**，并打印当前启用的后端。

这里设置 `config.PROBLEM = 'A'`，这样无论文件里写的是什么，本 notebook 都能按问题 A 跑。提交时请把这项写进 `config.py` 本身，不要依赖 notebook cell。

**看打印出来的那一行。** `BACKEND=scripted` 才让这次运行免费、离线、每次结果相同 —— 提交仓库里的默认值也必须是它。

**如果打印结果和 `config.py` 不一致**，脚手架会用大写字母警告 —— 那是 Python 在复用缓存的字节码。请重启 kernel 再跑一遍。


In [3]:
# --- 环境设置 ------------------------------------------------------
# 本地：本 cell 可直接运行。Colab：请改下面两条路径。
import os, sys, json

SCAFFOLD = '/Users/forstmac/Desktop/NTU/PE6201/A2/A2_scaffold'          # 例如 '/content/drive/MyDrive/PE6201/A2_scaffold'
# os.environ['A2_DATA'] = '/content/drive/MyDrive/PE6201/A2_reference_data'

sys.path.insert(0, SCAFFOLD)
import config
config.PROBLEM = 'A'            # 本 notebook 对应问题 A

print(config.summary())
print('data:', config.data_root())


BACKEND=scripted  FREE, deterministic  |  PROBLEM=A  |  model=(no model)  |  cap=8 turns  |  autonomy=confirm
data: /Users/forstmac/Desktop/NTU/PE6201/A2/A2_reference_data


---
## Cell 2 · 智能体拿到的是什么

Harness 只给智能体 **一个理赔 id，别的什么都没有**。本 cell 取出 `CLM-8842`，让你看清这条记录里到底有什么。

**先看它里面没有什么。** 没有保单。没有承保结论。没有网络医院状态。没有预授权。这些都要在运行过程中用工具去取。

**下面整条链路由三个字段驱动：**

- `member_id` 是 **`M-2214`** —— 查保单的入口，也是重复理赔核对的一半依据。
- `hospital_id` 是 **`H-114`** —— 网络医院状态。
- `lines` 有 **三条**。这是最关键的数字：**每一行都要单独做承保检查，并给出各自的处置。** 随包 15 条理赔里，9 条只有一行，6 条有两到四行。只检查第一行的智能体会悄悄批准本该拒赔的项目。


In [4]:
import tools

claim = tools.get_claim('CLM-8842')
# print(claim)
print(json.dumps(claim, indent=1))
print()
print('会员     :', claim['member_id'])
print('医院     :', claim['hospital_id'])
print('明细行   :', len(claim['lines']), '->',
      [l['code'] for l in claim['lines']])


{
 "claim_id": "CLM-8842",
 "member_id": "M-2214",
 "hospital_id": "H-114",
 "date_of_service": "2026-09-02",
 "narrative": "Admitted for appendix removal. Surgeon also treated a back problem and did a skin procedure while I was in.",
 "documents": [
  "itemised_bill",
  "discharge_summary"
 ],
 "lines": [
  {
   "code": "47120",
   "amount": 1400
  },
  {
   "code": "62480",
   "amount": 780
  },
  {
   "code": "31255",
   "amount": 300
  }
 ]
}

会员     : M-2214
医院     : H-114
明细行   : 3 -> ['47120', '62480', '31255']


---
## Cell 3 · 保单 —— 大多数拒赔从这里来

`lookup_policy('M-2214')` —— **用 `'M-2214'` 是因为那是本条理赔的会员**，见 cell 2 —— 路径是 理赔 → 会员 → 保单。两跳，第一跳本身完全不含决策信息。

**返回的这一行里藏着三种彼此独立的升级（escalation）理由**，很容易混为一谈：

1. `status == 'lapsed'` —— 其它都不用看了
2. 就诊日不在 `start_date .. end_date` 之内 —— **即使 status 显示 active 也一样**
3. 各行金额合计超过 `remaining`

**`remaining` = `annual_limit` 减去 `used_to_date`**，理赔总额要比的是 *这个余额*，不是年限额本身。拿年限额去比，在任何已有花费的保单上都会得到一个不出错提示的错误答案。

对本条理赔：有效、日期在保期内、总额远低于余额。所以三条都不触发，运行继续。


In [5]:
pol = tools.lookup_policy('M-2214')          # M-2214：见 cell 2
p = pol['policy']
#print(json.dumps(pol, indent=1))
#print(json.dumps(p, indent=1))
print('policy      :', p['policy_id'], p['product'])
print('status      :', p['status'])
print('cover dates :', p['start_date'], '..', p['end_date'],
       '   claim date:', claim['date_of_service'])
print('annual limit: %5d' % p['annual_limit'])
print('used to date: %5d' % p['used_to_date'])
print('REMAINING   : %5d   <- 理赔要比的是这个余额' % pol['remaining'])
print()
total = sum(l['amount'] for l in claim['lines'])
print('claim total : %5d  -> %s' % (total,
      '未超余额' if total <= pol['remaining'] else '超额 - 升级'))
print()
print('exclusions on this policy:', json.dumps(p['exclusions']))


policy      : POL-3310 Shield Plus
status      : active
cover dates : 2026-04-01 .. 2027-03-31    claim date: 2026-09-02
annual limit: 12000
used to date:  2800
REMAINING   :  9200   <- 理赔要比的是这个余额

claim total :  2480  -> 未超余额

exclusions on this policy: [{"code": "31255", "rule": "EX-14 cosmetic dermatology"}, {"code": "15823", "rule": "EX-14 cosmetic dermatology"}]


---
## Cell 4 · 三行明细 —— 以及驱动后续流程的两个字段

`check_coverage` **每一行调用一次**。三行就三次 —— 而且彼此独立，所以三次都应落在同一回合（turn）里。

下表给出决定下一步的两个字段：

**`requires_preauth` 就是那个分支。** 只有 `62480` 需要预授权。连调三次 `get_preauthorisation` 的智能体等于没读这个标志 —— 而没有任何一行需要预授权的理赔，会少一整回合。*这就是* 为什么不同理赔的回合数不同：分支不是你写的，是记录写的。

**`excluded` 拒的是这一行，不是整条索赔。** `31255` 按 EX-14 除外，但整单决策仍是 `approve_in_principle` —— 三行有的批、有的拒，写在 **同一封同时覆盖两者的决定函** 里。因为一行除外就把整单升级，是一种常见且独立的失败；记录必须点出规则名称，不能只写 “excluded”。


In [6]:
print('%-8s %-30s %-16s %-9s %s'
      % ('code', 'description', 'requires_preauth', 'excluded', 'rule'))
print('-' * 84)
for line in claim['lines']:
    cov = tools.check_coverage(line['code'], 'POL-3310')   # POL-3310：见 cell 3
    print('%-8s %-30s %-16s %-9s %s'
          % (cov['code'], cov['description'][:30],
             cov['requires_preauth'], cov['excluded'],
             cov['exclusion_rule'] or ''))
print()
print('-> 只有一行需要预授权，所以只调一次，不是三次')
print('-> 有一行除外：拒的是这一行，不是整条索赔')


code     description                    requires_preauth excluded  rule
------------------------------------------------------------------------------------
47120    Laparoscopic appendicectomy    False            False     
62480    Lumbar spinal fusion           True             False     
31255    Cosmetic dermabrasion          False            True      EX-14 cosmetic dermatology

-> 只有一行需要预授权，所以只调一次，不是三次
-> 有一行除外：拒的是这一行，不是整条索赔


---
## Cell 5 · 本案例里的陷阱

这条理赔 **不是** 重复件。但历史里有一条已决理赔，**就诊日相同**，会员不同、医院也不同。

因此，若智能体用少于 **全部四个事实** —— 会员、医院、就诊日、明细行 —— 去判重复，就会 **把这条完全没问题的理赔错误升级。**

下面的 cell 用四种匹配策略扫整条队列，并显示各自标出了哪些理赔。`CLM-8933` 是 **唯一** 真正的重复件；同一行里其它任何 id 都是假阳性 —— 好单被错升。

**只有最后一种策略是对的。** 只按日期匹配会错标 *本条* 理赔。另外两种捷径碰不到 `CLM-8842`，却会错标 `CLM-8960` —— 所以三种捷径都会失败，只是栽在不同单上。cell 会打印出来，不必凭信任。

历史里那三条「差一点」的记录是故意放的，好让捷径匹配被罚，而不是碰巧蒙对。

*（本 cell 深入调用 `tools._load` 给你看原始表。这是教学捷径 —— 你的智能体绝不能这么做。）*


In [7]:
claims  = tools._load('A', 'claims')
decided = tools._load('A', 'decided_claims')
norm = lambda ls: sorted((x['code'], x['amount']) for x in ls)

strategies = {
    '仅日期                  ': lambda c, d: c['date_of_service'] == d['date_of_service'],
    '会员 + 日期               ': lambda c, d: (c['member_id'], c['date_of_service'])
                                          == (d['member_id'], d['date_of_service']),
    '会员+医院+日期            ': lambda c, d: (c['member_id'], c['hospital_id'],
                                           c['date_of_service'])
                                          == (d['member_id'], d['hospital_id'],
                                              d['date_of_service']),
    '全部四个事实              ': lambda c, d: (c['member_id'], c['hospital_id'],
                                           c['date_of_service']) ==
                                          (d['member_id'], d['hospital_id'],
                                           d['date_of_service'])
                                          and norm(c['lines']) == norm(d['lines']),
}

for label, match in strategies.items():
    hits = sorted({c['claim_id'] for c in claims for d in decided if match(c, d)})
    ok = (hits == ['CLM-8933'])
    flag = '正确' if ok else '错误 - 假阳性'
    print('%s 标出 %-40s %s' % (label, ', '.join(hits), flag))
print()
# 算出来的，不是写死断言 —— 这样 notebook 不会和数据脱节。
this = 'CLM-8842'
wrong_here = [lbl.strip() for lbl, m in strategies.items()
              if any(m(c, d) for c in claims if c['claim_id'] == this for d in decided)]
wrong_anywhere = [lbl.strip() for lbl, m in strategies.items()
                  if sorted({c['claim_id'] for c in claims for d in decided
                             if m(c, d)}) != ['CLM-8933']]
print('%s —— 本 notebook 的这条理赔 —— 不是重复件。' % this)
print('  会错标本条理赔的策略                 : %s' % (', '.join(wrong_here) or '无'))
print('  会错标某条理赔的策略                 : %s' % ', '.join(wrong_anywhere))
print()
print('每种捷径都会在某处失败。只有四个事实全部匹配才是对的。')


仅日期                   标出 CLM-8842, CLM-8933, CLM-8960             错误 - 假阳性
会员 + 日期                标出 CLM-8933, CLM-8960                       错误 - 假阳性
会员+医院+日期             标出 CLM-8933, CLM-8960                       错误 - 假阳性
全部四个事实               标出 CLM-8933                                 正确

CLM-8842 —— 本 notebook 的这条理赔 —— 不是重复件。
  会错标本条理赔的策略                 : 仅日期
  会错标某条理赔的策略                 : 仅日期, 会员 + 日期, 会员+医院+日期

每种捷径都会在某处失败。只有四个事实全部匹配才是对的。


---
## Cell 6 · 真正跑一遍

现在整条链路按回合走。`verbose=True` 会打印模型的思考以及每次工具调用的结果。

**八次工具调用、四个回合** —— 这与附录 A 完全一致，值得自己核对，不要凭信任：

| 回合 | 调用 | 为何这样分组 |
|---|---|---|
| 1 | `get_claim` | 必须单独跑 —— 其它都需要会员、医院和明细行 |
| 2 | `lookup_policy` + `check_coverage` ×3 + `lookup_hospital` | **五次调用**，彼此独立 |
| 3 | `get_preauthorisation` | **不能**并进第 2 回合 —— 承保结果出来之前，不知道哪一行需要预授权 |
| 4 | `issue_decision_letter` | 受闸控的动作 —— 同样占一个回合 |

**第 3 回合把依赖规则摊开给你看。** 三次承保检查能折进同一回合，因为它们独立。预授权不行，因为它依赖那些答案。八次全串行就是八个回合；把那五次折进去就是四个 —— 约省 54% token，而 D2(c) 要你推理的正是这件事。

`thought` 文本是脚本写死的，不是生成的。用来展示模型在每一步 *会* 怎么想。


In [8]:
from agent import run_case

record = run_case('CLM-8842', problem='A', verbose=True)


  turn 1    · Turn 1 must run alone: everything else needs the member, the hospital and the LINE ITEMS
       get_claim                  -> {'claim_id': 'CLM-8842', 'member_id': 'M-2214', 'hospital_id': …
  turn 2    · Now five calls that depend on nothing but that record. The policy, the hospital, and one
       lookup_policy              -> {'member': {'member_id': 'M-2214', 'name': 'Tan Wei Ling', 'pol…
       check_coverage             -> {'code': '47120', 'description': 'Laparoscopic appendicectomy',…
       check_coverage             -> {'code': '31255', 'description': 'Cosmetic dermabrasion', 'requ…
       check_coverage             -> {'code': '62480', 'description': 'Lumbar spinal fusion', 'requi…
       lookup_hospital            -> {'hospital_id': 'H-114', 'name': 'Riverside General', 'panel': …
  turn 3    · This one CANNOT join the turn above: I did not know which line needed a pre-authorisatio
       get_preauthorisation       -> {'preauth_id': 'PA-5521', 'member_id': 'M-

---
## Cell 7 · 决策记录

这才是被评分的东西。值得分成两半看：

**答案** —— `decision` 和 `reason`。注意 reason 里对 *每一行* 都有处置，还有两个合计，以及除外规则的名称。

**观测数据** —— `turns`、`tokens_in`、`cost_usd`、`evidence`、`guardrails_fired`。是在 *运行过程中* 捕获的，因为你无法报告自己根本没机会发现的失败。D6 的成本模型和 D7 的循环失败都需要这些。

注意 `guardrails_fired` 对 `issue_decision_letter` 显示 `gate_passed` —— 不可逆步骤经过了自主权闸门，记录可以证明。

**token 数字是估算值**，因为脚本后端没有真实模型。D6 要的是 *实测* 计数，那意味着要用线上电池（live battery）。


In [9]:
print(json.dumps(record, indent=2))


{
  "decision": "approve_in_principle",
  "reason": "3 lines. 47120 covered (1400). 62480 covered, PA-5521 cited, valid on 2026-09-02 (780). 31255 refused under EX-14 cosmetic dermatology (300). approved_total 2180, refused_total 300. H-114 is on panel.",
  "case_id": "CLM-8842",
  "evidence": [
    "get_claim",
    "lookup_policy",
    "check_coverage",
    "check_coverage",
    "check_coverage",
    "lookup_hospital",
    "get_preauthorisation",
    "issue_decision_letter"
  ],
  "turns": 4,
  "tokens_in": 21000,
  "tokens_out": 600,
  "cost_usd": 0.00234,
  "seconds": 0.001,
  "guardrails_fired": [
    {
      "guardrail": "gate_passed",
      "detail": "issue_decision_letter (autonomy=confirm)"
    }
  ],
  "stopped_by": null,
  "backend": "scripted"
}


---
## Cell 8 · 评分 —— 代码核对（code check）

对照标准答案做确定性比较。**没有模型、没有人、没有主观意见。** 分数就是从这里来的。

先打印 `CLM-8842` 对应的标准答案行。注意它没有 `booked` 字段 —— 那是问题 B 才有的；问题 A 从不预订任何东西。

**不比较** 的：措辞、回合数、成本。两个智能体都可以答对，但花费差很多，那是 D6 的主题。


In [10]:
from harness import load_key, code_check, prepare_judgement_check

expected = load_key('A')['CLM-8842']
print('标准答案写的是:')
print(json.dumps(expected, indent=1))

passed, fails = code_check(record, expected)
print()
print('CODE CHECK:', 'PASS' if passed else 'FAIL')
for f in fails:
    print('   ', f)


标准答案写的是:
{
 "case_id": "CLM-8842",
 "expected_decision": "approve_in_principle",
 "family": "partly_payable",
 "must_record": [
  "a disposition for all 3 lines",
  "31255 refused under EX-14 cosmetic dermatology",
  "PA-5521 cited for line 62480",
  "approved_total 2180",
  "refused_total 300"
 ],
 "note": "The brief's worked example. Not an approve and not a decline: one decision letter covering both."
}

CODE CHECK: PASS


---
## Cell 9 · 评分 —— 判断核对（judgement check）

**这是通过率看不出来的那一半。** 三种可能结果下，抛硬币单靠 code check 也能拿 33%；智能体也可以用错误理由走到正确决策，code check 发现不了。

对本条理赔，`must_record` 要求五件事 —— 包括 *全部 3 行各自的处置*，以及卡住 `31255` 的那条除外。只写 “approved” 的智能体会过 code check，却在这里失败，而这正是要点。

`prepare_judgement_check` **只组队列，不下结论**。有人读 reason 并对每一项裁定：可以是人，也可以是第二个模型。检查的种类相同，差别只在谁打分。若用模型自动评，请在报告里写明 —— 模型给模型打分，是一个需要辩护的主张。


In [11]:
item = prepare_judgement_check(record, expected)

print('智能体给出的 reason:')
print('   ', item['reason'])
print()
print('是否包含下面每一项？目前还没有人裁定:')
for m in item['must_record']:
    print('  [ ]', m)
print()
print('verdict:', item['verdict'], '   graded_by:', item['graded_by'])


智能体给出的 reason:
    3 lines. 47120 covered (1400). 62480 covered, PA-5521 cited, valid on 2026-09-02 (780). 31255 refused under EX-14 cosmetic dermatology (300). approved_total 2180, refused_total 300. H-114 is on panel.

是否包含下面每一项？目前还没有人裁定:
  [ ] a disposition for all 3 lines
  [ ] 31255 refused under EX-14 cosmetic dermatology
  [ ] PA-5521 cited for line 62480
  [ ] approved_total 2180
  [ ] refused_total 300

verdict: None    graded_by: None


---
## Cell 10 · 不会抛异常的失败  (D7)

同一案例、同一数据。**删掉一道护栏** —— 动作去重 —— 其它一律不变。这正是 D7 要求的形态：*能工作的智能体，减去 X*。把 X 加回去行为就恢复，这才叫诊断，而不是讲故事。

盯住 BEFORE / AFTER 两行里的三个数字：

- **turns** 4 → 6
- **cost** 大约高 1.8 倍
- **decision** 不变 —— 仍是 `approve_in_principle`

**没有异常。没有报错。答案还是对的。** 通过率表会把这次跑分记成干净通过。步数上限（8 回合）和预算上限（60,000 tokens）也都不会触发，因为都没被突破 —— 它们限制损害，并不检测故障。

你只有 **在计数** 的时候才会看见这件事。这就是 D7 的全部教训。


In [12]:
import demo_loop_failure
demo_loop_failure.main(case='CLM-8842', problem='A')



BACKEND=scripted  FREE, deterministic  |  PROBLEM=A  |  model=(no model)  |  cap=8 turns  |  autonomy=confirm
  demonstrating on CLM-8842 (Problem A)

BEFORE - the working agent, guard in place
  turns 4 · tool calls 8 · tokens 21600 · cost US$0.00234 · decision approve_in_principle

AFTER - the working agent MINUS action de-duplication
  turns 6 · tool calls 18 · tokens 38640 · cost US$0.00412 · decision approve_in_principle
  stopped by: None

  1 · THE INSTRUMENTATION THAT FOUND IT
      turns and cost logged per run. NOTHING RAISED AN EXCEPTION.
      The run cost 1.8x more and still answered 'approve_in_principle'
      - THE SAME ANSWER AS THE WORKING AGENT. A pass-rate table
      alone would show this run as a clean pass. It is only
      visible because turns and cost were counted.
  2 · THE TURN DISTRIBUTION
      before: 4 turns   after: 6 turns   cap: 8
      runs that hit the cap: 0 of 2
  3 · THE FIX, AND WHY THE OTHER TWO LAYERS WERE WRONG
      Action de-duplication ca

---
## Cell 11 · 现在改成你自己的

上面全部只跑了 **一条脚本化案例**。你的集合需要 30–50 条。

1. **`config.py`** —— 在文件里设置 `PROBLEM = 'A'`，不要只写在这个 notebook 里。
2. **`tools.py`** —— 先读每个工具上的注释块，再读底部的六字段描述符。运行 **`python3 run_eval.py --prompt`**，看这些描述符如何变成发给模型的文本。然后故意写一套更差的：那就是你 D2(b) 的 **v1**，可测量的对比才是交付物。
3. **`backends.py`** —— 自己再脚本化第二条理赔。**如果写不出步骤，说明你还没懂这个案例。** 现在发现，总好过 13 号凌晨两点才发现。`CLM-8933` —— 那条重复件 —— 是很好的第二条。
4. **`expected_outcomes_A.json`** —— 每加一条案例都要先打标签，**对照附录 A 的路由表，而且要在对它跑智能体之前**。见 `PE6201_A2_Adding_Extra_Cases.pdf`。
5. **`guardrails.py`** —— 根据证据设上限。若中位数是 4 回合、最差的合法跑分是 7，上限 8 站得住脚，上限 30 只是装饰。

然后关掉这个 notebook，到模块里干活。

```bash
python3 run_eval.py
```
